In [1]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm
import re
import warnings
warnings.filterwarnings('ignore')

print("✅ Imports ready")

✅ Imports ready


In [2]:
# Load job postings
jobs = pd.read_csv("C:/Users/prakh/connect-ml/data/raw/postings.csv")

# Load skills
skills = pd.read_csv("C:/Users/prakh/connect-ml/data/raw/job_skills.csv")

print("=== JOB POSTINGS ===")
print(f"Shape: {jobs.shape}")
print(f"Columns: {jobs.columns.tolist()}\n")

print("=== JOB SKILLS ===")
print(f"Shape: {skills.shape}")
print(f"Columns: {skills.columns.tolist()}\n")

# Preview skills file
print("Skills file sample:")
skills.head(10)

=== JOB POSTINGS ===
Shape: (123849, 31)
Columns: ['job_id', 'company_name', 'title', 'description', 'max_salary', 'pay_period', 'location', 'company_id', 'views', 'med_salary', 'min_salary', 'formatted_work_type', 'applies', 'original_listed_time', 'remote_allowed', 'job_posting_url', 'application_url', 'application_type', 'expiry', 'closed_time', 'formatted_experience_level', 'skills_desc', 'listed_time', 'posting_domain', 'sponsored', 'work_type', 'currency', 'compensation_type', 'normalized_salary', 'zip_code', 'fips']

=== JOB SKILLS ===
Shape: (213768, 2)
Columns: ['job_id', 'skill_abr']

Skills file sample:


,job_id,skill_abr
0,3884428798,MRKT
1,3884428798,PR
2,3884428798,WRT
3,3887473071,SALE
4,3887465684,FIN
5,3887465684,SALE
6,3887467939,SALE
7,3887467939,ADVR
8,3887467939,BD
9,3887471331,ENG


In [3]:
# Keep only what we need from jobs
jobs_slim = jobs[[
    'job_id', 'title', 'description', 'formatted_experience_level'
]].copy()

# Drop rows with no title or description
jobs_slim = jobs_slim.dropna(subset=['title', 'description'])

print(f"Jobs before merge : {len(jobs_slim):,}")

# Merge — each job_id may have multiple skill rows, so group them first
skills_grouped = skills.groupby('job_id')['skill_abr'].apply(
    lambda x: ', '.join(x.dropna().astype(str))
).reset_index()
skills_grouped.columns = ['job_id', 'skills_list']

print(f"Jobs with skills  : {len(skills_grouped):,}")

# Merge jobs with skills
df_merged = jobs_slim.merge(skills_grouped, on='job_id', how='left')
df_merged['skills_list'] = df_merged['skills_list'].fillna('')

print(f"After merge       : {len(df_merged):,}")
print(f"\nSample row:")
print(df_merged.iloc[0])

Jobs before merge : 123,842
Jobs with skills  : 126,807
After merge       : 123,842

Sample row:
job_id                                                                   921716
title                                                     Marketing Coordinator
description                   Job descriptionA leading real estate firm in N...
formatted_experience_level                                                  NaN
skills_list                                                          MRKT, SALE
Name: 0, dtype: object


In [4]:
def build_job_text(row):
    """Combine title, skills, and description into one rich text."""
    title = str(row['title']).strip()
    skills = str(row['skills_list']).strip()
    # Use first 400 chars of description to keep it fast
    desc = str(row['description'])[:400].strip()
    
    if skills:
        return f"Job Title: {title}. Required Skills: {skills}. Description: {desc}"
    else:
        return f"Job Title: {title}. Description: {desc}"

df_merged['job_text'] = df_merged.apply(build_job_text, axis=1)

print("Sample job_text:")
print(df_merged['job_text'].iloc[0])
print(f"\n✅ Built job_text for {len(df_merged):,} jobs")

Sample job_text:
Job Title: Marketing Coordinator. Required Skills: MRKT, SALE. Description: Job descriptionA leading real estate firm in New Jersey is seeking an administrative Marketing Coordinator with some experience in graphic design. You will be working closely with our fun, kind, ambitious members of the sales team and our dynamic executive team on a daily basis. This is an opportunity to be part of a fast-growing, highly respected real estate brokerage with a reputation for except

✅ Built job_text for 123,842 jobs


In [5]:
print("Loading model...")
model = SentenceTransformer('all-MiniLM-L6-v2')
print("✅ Model loaded")

# Each domain is described in natural language
# The model will compare every job against these and assign the closest one
DOMAIN_DESCRIPTIONS = {
    "Software Engineering": 
        "Software engineering, backend development, frontend development, "
        "full stack, mobile development, DevOps, cloud infrastructure, "
        "API development, system design, Java, Python, JavaScript, C++",
    
    "Data & AI": 
        "Data science, machine learning, artificial intelligence, data analysis, "
        "data engineering, business intelligence, statistics, SQL, Python, "
        "deep learning, NLP, computer vision, analytics, data pipelines",
    
    "Product Management": 
        "Product management, product strategy, roadmap planning, user research, "
        "agile, scrum, product lifecycle, go to market, stakeholder management, "
        "product metrics, growth, product vision",
    
    "Design": 
        "UX design, UI design, product design, user experience, user interface, "
        "interaction design, visual design, wireframing, prototyping, Figma, "
        "user research, design systems, accessibility",
    
    "Finance": 
        "Investment banking, financial analysis, equity research, accounting, "
        "financial modeling, valuation, portfolio management, risk management, "
        "consulting, strategy, Excel, financial planning",
    
    "Marketing": 
        "Digital marketing, content marketing, brand management, SEO, SEM, "
        "social media, growth hacking, email marketing, performance marketing, "
        "market research, campaign management, customer acquisition",
    
    "Human Resources":
        "Human resources, talent acquisition, recruiting, people operations, "
        "employee relations, compensation, benefits, HR business partner, "
        "organizational development, culture, onboarding",
    
    "Sales & Business Development":
        "Sales, business development, account management, B2B sales, "
        "enterprise sales, revenue growth, CRM, Salesforce, partnerships, "
        "client relations, sales strategy, lead generation"
}

# Encode all domain descriptions
domain_names = list(DOMAIN_DESCRIPTIONS.keys())
domain_texts = list(DOMAIN_DESCRIPTIONS.values())
domain_embeddings = model.encode(domain_texts)

print(f"✅ {len(domain_names)} domains defined and encoded")
print("Domains:", domain_names)

Loading model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Model loaded
✅ 8 domains defined and encoded
Domains: ['Software Engineering', 'Data & AI', 'Product Management', 'Design', 'Finance', 'Marketing', 'Human Resources', 'Sales & Business Development']


In [6]:
print(f"Encoding {len(df_merged):,} jobs... (this takes 5-8 mins)\n")

# Encode all jobs
job_embeddings_full = model.encode(
    df_merged['job_text'].tolist(),
    batch_size=64,
    show_progress_bar=True
)

print("\n✅ Encoding complete")
print(f"Embedding shape: {job_embeddings_full.shape}")

Encoding 123,842 jobs... (this takes 5-8 mins)



Batches:   0%|          | 0/1936 [00:00<?, ?it/s]


✅ Encoding complete
Embedding shape: (123842, 384)


In [7]:
# Compare each job to each domain and pick the best match
similarities = cosine_similarity(job_embeddings_full, domain_embeddings)

# Get index of highest scoring domain for each job
best_domain_indices = similarities.argmax(axis=1)
best_domain_scores = similarities.max(axis=1)

# Assign domain and confidence score
df_merged['domain'] = [domain_names[i] for i in best_domain_indices]
df_merged['domain_confidence'] = best_domain_scores.round(3)

print("Domain distribution (ALL jobs classified):")
print(df_merged['domain'].value_counts())
print(f"\nAverage confidence: {df_merged['domain_confidence'].mean():.3f}")
print(f"Low confidence jobs (< 0.3): {(df_merged['domain_confidence'] < 0.3).sum():,}")

Domain distribution (ALL jobs classified):
domain
Human Resources                 74640
Sales & Business Development    23387
Software Engineering             7153
Finance                          5811
Data & AI                        4340
Product Management               3416
Design                           2555
Marketing                        2540
Name: count, dtype: int64

Average confidence: 0.270
Low confidence jobs (< 0.3): 81,698


In [8]:
# Keep only jobs where the model is reasonably confident
df_final = df_merged[df_merged['domain_confidence'] >= 0.3].copy()

print(f"Before confidence filter : {len(df_merged):,}")
print(f"After confidence filter  : {len(df_final):,}")
print(f"Dropped                  : {len(df_merged) - len(df_final):,}")
print(f"\nFinal domain distribution:")
print(df_final['domain'].value_counts())

# Save
df_final.to_csv(
    "C:/Users/prakh/connect-ml/data/processed/jobs_full_classified.csv",
    index=False
)
print("\n✅ Saved to data/processed/jobs_full_classified.csv")

Before confidence filter : 123,842
After confidence filter  : 42,144
Dropped                  : 81,698

Final domain distribution:
domain
Human Resources                 17606
Sales & Business Development    11665
Software Engineering             4051
Finance                          3457
Data & AI                        2169
Product Management               1372
Marketing                        1143
Design                            681
Name: count, dtype: int64

✅ Saved to data/processed/jobs_full_classified.csv


In [9]:
# Save embeddings for the final classified jobs only
final_mask = df_merged['domain_confidence'] >= 0.3
embeddings_final = job_embeddings_full[final_mask]

np.save(
    "C:/Users/prakh/connect-ml/models/job_embeddings_full.npy",
    embeddings_final
)

print(f"✅ Saved embeddings shape: {embeddings_final.shape}")
print("\nAll files saved:")
print("  data/processed/jobs_full_classified.csv")
print("  models/job_embeddings_full.npy")

✅ Saved embeddings shape: (42144, 384)

All files saved:
  data/processed/jobs_full_classified.csv
  models/job_embeddings_full.npy


In [10]:
print("=== FINAL DATASET SUMMARY ===\n")
print(f"Total jobs: {len(df_final):,}\n")

print("Domain distribution:")
dist = df_final['domain'].value_counts()
for domain, count in dist.items():
    bar = '█' * (count // 200)
    print(f"  {domain:<35} {count:>6,}  {bar}")

print(f"\nAverage confidence score : {df_final['domain_confidence'].mean():.3f}")
print(f"Min confidence           : {df_final['domain_confidence'].min():.3f}")
print(f"Max confidence           : {df_final['domain_confidence'].max():.3f}")

print("\nExperience level distribution:")
print(df_final['formatted_experience_level'].value_counts())

=== FINAL DATASET SUMMARY ===

Total jobs: 42,144

Domain distribution:
  Human Resources                     17,606  ████████████████████████████████████████████████████████████████████████████████████████
  Sales & Business Development        11,665  ██████████████████████████████████████████████████████████
  Software Engineering                 4,051  ████████████████████
  Finance                              3,457  █████████████████
  Data & AI                            2,169  ██████████
  Product Management                   1,372  ██████
  Marketing                            1,143  █████
  Design                                 681  ███

Average confidence score : 0.361
Min confidence           : 0.300
Max confidence           : 0.673

Experience level distribution:
formatted_experience_level
Mid-Senior level    17174
Entry level          8655
Associate            3668
Director             1655
Executive             592
Internship            405
Name: count, dtype: int64


In [11]:
# Cap each domain to 4000 jobs max, keep best confidence scores
DOMAIN_CAP = 4000

df_balanced = (
    df_final
    .sort_values('domain_confidence', ascending=False)  # best first
    .groupby('domain')
    .head(DOMAIN_CAP)
    .reset_index(drop=True)
)

print("Balanced domain distribution:")
dist = df_balanced['domain'].value_counts()
for domain, count in dist.items():
    bar = '█' * (count // 100)
    print(f"  {domain:<35} {count:>6,}  {bar}")

print(f"\nTotal jobs after balancing: {len(df_balanced):,}")

Balanced domain distribution:
  Human Resources                      4,000  ████████████████████████████████████████
  Sales & Business Development         4,000  ████████████████████████████████████████
  Software Engineering                 4,000  ████████████████████████████████████████
  Finance                              3,457  ██████████████████████████████████
  Data & AI                            2,169  █████████████████████
  Product Management                   1,372  █████████████
  Marketing                            1,143  ███████████
  Design                                 681  ██████

Total jobs after balancing: 20,822


In [12]:
# Only keep jobs where the model was confident in its classification
df_confident = df_balanced[df_balanced['domain_confidence'] >= 0.35].copy()

print(f"Before confidence filter : {len(df_balanced):,}")
print(f"After confidence filter  : {len(df_confident):,}")
print(f"\nDomain distribution after filtering:")
print(df_confident['domain'].value_counts())
print(f"\nAverage confidence: {df_confident['domain_confidence'].mean():.3f}")

Before confidence filter : 20,822
After confidence filter  : 15,641

Domain distribution after filtering:
domain
Human Resources                 4000
Sales & Business Development    4000
Software Engineering            2402
Finance                         2090
Data & AI                       1441
Product Management               741
Marketing                        713
Design                           254
Name: count, dtype: int64

Average confidence: 0.416


In [13]:
print(f"Rebuilding embeddings for {len(df_confident):,} jobs...\n")

embeddings_clean = model.encode(
    df_confident['job_text'].tolist(),
    batch_size=64,
    show_progress_bar=True
)

print(f"\n✅ Embeddings shape: {embeddings_clean.shape}")

Rebuilding embeddings for 15,641 jobs...



Batches:   0%|          | 0/245 [00:00<?, ?it/s]


✅ Embeddings shape: (15641, 384)


In [14]:
# Save balanced confident dataset
df_confident.to_csv(
    "C:/Users/prakh/connect-ml/data/processed/jobs_balanced.csv",
    index=False
)

# Save new embeddings
np.save(
    "C:/Users/prakh/connect-ml/models/job_embeddings_balanced.npy",
    embeddings_clean
)

print("✅ Saved:")
print(f"   data/processed/jobs_balanced.csv       ({len(df_confident):,} jobs)")
print(f"   models/job_embeddings_balanced.npy     {embeddings_clean.shape}")

✅ Saved:
   data/processed/jobs_balanced.csv       (15,641 jobs)
   models/job_embeddings_balanced.npy     (15641, 384)


In [15]:
def test_predictor(skills, interests, domain):
    student_profile = f"Skills: {skills}. Interests: {interests}"
    student_emb = model.encode([student_profile])
    
    mask = df_confident['domain'] == domain
    domain_df = df_confident[mask].reset_index(drop=True)
    domain_embs = embeddings_clean[mask]
    
    sims = cosine_similarity(student_emb, domain_embs)[0]
    top_idx = sims.argsort()[::-1][:15]
    
    seen, results = set(), []
    for idx in top_idx:
        title = re.sub(r'\s+\d+$', '', domain_df.iloc[idx]['title'].title().strip())
        title = re.sub(r'\s+(Ii|Iii|Iv|Vi|Vii)$', '', title).strip()
        score = round(float(sims[idx]) * 100, 1)
        exp = domain_df.iloc[idx]['formatted_experience_level']
        if pd.isna(exp): exp = 'All levels'
        
        if title not in seen:
            seen.add(title)
            results.append((title, score, exp))
        if len(results) == 3:
            break
    return results

# Test all domains
tests = [
    ("Python, SQL, machine learning", "building AI models", "Data & AI"),
    ("JavaScript, React, Node.js", "building web apps", "Software Engineering"),
    ("communication, strategy, leadership", "managing products", "Product Management"),
    ("Figma, wireframing, user research", "designing interfaces", "Design"),
    ("Excel, financial modeling, valuation", "analyzing investments", "Finance"),
    ("SEO, content, social media", "growing brands online", "Marketing"),
    ("recruiting, onboarding, culture", "helping people grow", "Human Resources"),
    ("B2B sales, CRM, negotiation", "closing deals and partnerships", "Sales & Business Development"),
]

print("🎯 QUALITY TEST — All Domains\n")
for skills, interests, domain in tests:
    results = test_predictor(skills, interests, domain)
    print(f"{'─'*50}")
    print(f"Domain : {domain}")
    print(f"Skills : {skills}")
    for i, (title, score, exp) in enumerate(results, 1):
        print(f"  {i}. {title} ({score}%) — {exp}")
print(f"{'─'*50}")

🎯 QUALITY TEST — All Domains

──────────────────────────────────────────────────
Domain : Data & AI
Skills : Python, SQL, machine learning
  1. Ai/Ml Solutions Architect (Data Analysis Skills, Sql/Big Query, Python) (65.8%) — Mid-Senior level
  2. Python Developer With Ai/Ml Skills (64.1%) — Entry level
  3. Data Scientist (63.1%) — Entry level
──────────────────────────────────────────────────
Domain : Software Engineering
Skills : JavaScript, React, Node.js
  1. Javascript Developer (71.6%) — Mid-Senior level
  2. Frontend Developer (70.4%) — All levels
  3. User Interface Architect (68.4%) — All levels
──────────────────────────────────────────────────
Domain : Product Management
Skills : communication, strategy, leadership
  1. Product Owner (68.1%) — Mid-Senior level
  2. Product Manager (66.4%) — Mid-Senior level
  3. Senior Product Manager (63.5%) — All levels
──────────────────────────────────────────────────
Domain : Design
Skills : Figma, wireframing, user research
  1. Ux/Ui

In [16]:
# Filter out job titles that are too long (they're usually messy/concatenated)
MAX_TITLE_LENGTH = 60

df_final_clean = df_confident[
    df_confident['title'].str.len() <= MAX_TITLE_LENGTH
].copy()

print(f"Before title filter : {len(df_confident):,}")
print(f"After title filter  : {len(df_final_clean):,}")
print(f"Removed             : {len(df_confident) - len(df_final_clean):,} messy titles")

# Rebuild embeddings for clean set
print(f"\nRebuilding embeddings...")
embeddings_final_clean = model.encode(
    df_final_clean['job_text'].tolist(),
    batch_size=64,
    show_progress_bar=True
)

# Save both
df_final_clean.to_csv(
    "C:/Users/prakh/connect-ml/data/processed/jobs_production.csv",
    index=False
)
np.save(
    "C:/Users/prakh/connect-ml/models/job_embeddings_production.npy",
    embeddings_final_clean
)

print(f"\n✅ Production dataset ready")
print(f"   Jobs              : {len(df_final_clean):,}")
print(f"   Embeddings shape  : {embeddings_final_clean.shape}")
print(f"\nFinal domain distribution:")
print(df_final_clean['domain'].value_counts())

Before title filter : 15,641
After title filter  : 15,058
Removed             : 583 messy titles

Rebuilding embeddings...


Batches:   0%|          | 0/236 [00:00<?, ?it/s]


✅ Production dataset ready
   Jobs              : 15,058
   Embeddings shape  : (15058, 384)

Final domain distribution:
domain
Sales & Business Development    3872
Human Resources                 3869
Software Engineering            2307
Finance                         1988
Data & AI                       1378
Product Management               698
Marketing                        698
Design                           248
Name: count, dtype: int64
